# Marketing Campaign dataset analysis

The client’s core question remains:

“Which campaign + channel combo should the client double down on to grow new customer sales—and why?”

**1. Load the dataset.**

In [4]:
from google.colab import files

uploaded = files.upload()

Saving Campaign_Data_Week1.xlsx to Campaign_Data_Week1 (1).xlsx


In [5]:
import pandas as pd

df = pd.read_excel("Campaign_Data_Week1.xlsx")
df.sample(5)

,Interaction ID,Campaign Type,Channel,Customer Type,"Converted (1=yes, 0=no)",Time on Site (seconds),Sales ($)
220,INT0221,B,Instagram,New,0,41.28,0.00
437,INT0438,B,Web Banner,Existing,0,29.79,0.00
238,INT0239,A,Instagram,New,1,90.89,87.09
475,INT0476,B,Email,Existing,1,118.51,57.62
260,INT0261,A,Instagram,Existing,1,116.35,50.88


In [ ]:
df.columns.to_list()

['Interaction ID',
 'Campaign Type',
 'Channel',
 'Customer Type',
 'Converted (1=yes, 0=no)',
 'Time on Site (seconds)',
 'Sales ($)']

**2. Inspect dataset**

In [6]:

# Dataset dimensions
print("Shape:", df.shape)
print(50*"*")

# Column information
df.info()
print(50*"*")

# Missing values
print(df.isnull().sum())
print(50*"*")

Shape: (500, 7)
**************************************************
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Interaction ID           500 non-null    object 
 1   Campaign Type            500 non-null    object 
 2   Channel                  500 non-null    object 
 3   Customer Type            500 non-null    object 
 4   Converted (1=yes, 0=no)  500 non-null    int64  
 5   Time on Site (seconds)   500 non-null    float64
 6   Sales ($)                500 non-null    float64
dtypes: float64(2), int64(1), object(4)
memory usage: 27.5+ KB
**************************************************
Interaction ID             0
Campaign Type              0
Channel                    0
Customer Type              0
Converted (1=yes, 0=no)    0
Time on Site (seconds)     0
Sales ($)                  0
dtype: int64
*************

In [7]:
df.describe()

,"Converted (1=yes, 0=no)",Time on Site (seconds),Sales ($)
count,500.000000,500.000000,500.000000
mean,0.478000,72.445660,29.319200
std,0.500016,36.757196,32.011189
min,0.000000,5.200000,0.000000
25%,0.000000,42.707500,0.000000
50%,0.000000,57.990000,0.000000
75%,1.000000,104.195000,60.900000
max,1.000000,175.490000,92.190000


**3. Create a pivot table:**

Pivot tables are a powerful way to summarize large datasets.

They let you group, filter, and compare results quickly without having to write formulas.

- Total Sales by Channel

- Total Sales by Campaign

In [13]:
sales_pivot_tab= pd.pivot_table(df, values="Sales ($)", index="Campaign Type", columns="Channel", aggfunc="sum", fill_value=0)
sales_pivot_tab

Channel,Email,Instagram,Web Banner
Campaign Type,,,
A,2952.37,2162.90,754.29
B,4956.94,1710.51,2122.59


**Insight:**

1. Email had the highest overall sales, making it the strongest performer in terms of revenue generation. It’s a reliable pick when ROI is a key concern.

2. Campaign B’s more promotional messaging style led to higher total sales, showing it was broadly effective across audiences.

3. When combining top-performing elements, Email + Campaign B stood out as the highest-revenue pairing — strong channel and strong message.

**4. Calculate average time on site:**

In [ ]:

time_pivot = pd.pivot_table(
    df,
    values="Time on Site (seconds)",
    index="Campaign Type",
    columns="Channel",
    aggfunc="mean"
)

time_pivot


Channel,Email,Instagram,Web Banner
Campaign Type,,,
A,85.762152,66.042289,60.569107
B,83.922302,62.179688,65.440543


recreate the slicer

In [15]:
new_customers = df[df["Customer Type"] == "New"]
new_customers.sample(5)

,Interaction ID,Campaign Type,Channel,Customer Type,"Converted (1=yes, 0=no)",Time on Site (seconds),Sales ($)
4,INT0005,A,Instagram,New,0,59.30,0.0
208,INT0209,B,Email,New,1,126.17,54.5
463,INT0464,A,Instagram,New,0,53.57,0.0
150,INT0151,B,Instagram,New,0,18.12,0.0
394,INT0395,B,Web Banner,New,0,36.01,0.0


In [16]:
new_sales_pivot = pd.pivot_table(new_customers,values="Sales ($)",index="Campaign Type",columns="Channel",aggfunc="sum",fill_value=0)
new_sales_pivot


Channel,Email,Instagram,Web Banner
Campaign Type,,,
A,2124.47,969.46,633.06
B,1382.38,416.14,318.92


**Campaign A (informal tone) combined with Email (personal channel) proved most effective in converting new customers – a key insight for future growth.**


In [17]:

# Create a function to analyse campaign performance

def campaign_analysis(data):

    result = data.groupby(
        ["Campaign Type", "Channel"]
    ).agg(
        Total_Sales=("Sales ($)", "sum"),
        Total_Interactions=("Interaction ID", "count"),
        Conversions=("Converted (1=yes, 0=no)", "sum"),
        Avg_Time=("Time on Site (seconds)", "mean")
    ).reset_index()

    result["Conversion_Rate"] = (
        result["Conversions"] /
        result["Total_Interactions"] * 100
    )

    return result.sort_values(
        "Total_Sales", ascending=False
    )


# Analyse all customers
overall = campaign_analysis(df)

# Analyse new customers
new_customers = df[df["Customer Type"] == "New"]

new_analysis = campaign_analysis(new_customers)

#print("OVERALL PERFORMANCE")
#display(overall.round(2))

print("NEW CUSTOMER PERFORMANCE")
display(new_analysis.round(2))


NEW CUSTOMER PERFORMANCE


,Campaign Type,Channel,Total_Sales,Total_Interactions,Conversions,Avg_Time,Conversion_Rate
0,A,Email,2124.47,47,29,94.99,61.70
3,B,Email,1382.38,31,20,90.34,64.52
1,A,Instagram,969.46,48,16,60.15,33.33
2,A,Web Banner,633.06,43,14,58.69,32.56
4,B,Instagram,416.14,20,7,51.44,35.00
5,B,Web Banner,318.92,24,7,65.66,29.17


Given that Email + Campaign A worked best for new customers, what should we consider doing next?


In [10]:
non_converted = df[
    df["Converted (1=yes, 0=no)"] == 0
].copy()

In [11]:
pivot_table = pd.pivot_table(
    non_converted,
    values="Time on Site (seconds)",
    index="Channel",
    columns="Campaign Type",
    aggfunc="mean",
    margins=True,
    margins_name="Grand Total"
)

print(pivot_table.round(2))

Campaign Type      A      B  Grand Total
Channel                                 
Email          42.97  39.43        40.89
Instagram      48.83  39.71        44.98
Web Banner     44.15  39.46        41.59
Grand Total    45.59  39.51        42.42
